# NSL-KDD Research Execution

Runs the final NSL-KDD FW-LNSA research profile with FS-10, FS-20, three matching methods, controlled thresholds, detector budgets, and five seeds.

Experiment outputs are stored in Google Drive so Colab disconnects do not erase completed runs.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Repository access

Store a GitHub personal access token in Colab Secrets with the name `GITHUB_TOKEN`. The token needs read access to the private repository. The temporary credential helper avoids placing the token in the Git remote URL.

In [ ]:
import os
import stat
import subprocess
from pathlib import Path
from google.colab import userdata

REPO_URL = "https://github.com/sarosh-jawed/FW-LNSA-NIDS.git"
REPO_DIR = Path("/content/FW-LNSA-NIDS")
TOKEN = userdata.get("GITHUB_TOKEN")
if not TOKEN:
    raise RuntimeError("Add GITHUB_TOKEN to Colab Secrets before continuing.")

askpass = Path("/tmp/fw_lnsa_git_askpass.sh")
askpass.write_text(
    "#!/bin/sh\n"
    "case \"$1\" in\n"
    "  *Username*) echo \"x-access-token\" ;;\n"
    "  *Password*) echo \"$GITHUB_TOKEN\" ;;\n"
    "esac\n"
)
askpass.chmod(askpass.stat().st_mode | stat.S_IXUSR)

environment = os.environ.copy()
environment["GITHUB_TOKEN"] = TOKEN
environment["GIT_ASKPASS"] = str(askpass)
environment["GIT_TERMINAL_PROMPT"] = "0"

if REPO_DIR.exists():
    subprocess.run(["git", "checkout", "main"], cwd=REPO_DIR, check=True, env=environment)
    subprocess.run(["git", "pull", "origin", "main"], cwd=REPO_DIR, check=True, env=environment)
else:
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True, env=environment)

commit = subprocess.run(
    ["git", "rev-parse", "HEAD"],
    cwd=REPO_DIR,
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()
print("Git commit:", commit)

In [ ]:
%cd /content/FW-LNSA-NIDS
!python -m pip install -q -r requirements.txt
!python scripts/validate_core_modules.py
!python scripts/validate_fw_lnsa_pipeline.py
!python scripts/validate_cicids2017_pipeline.py
!python scripts/validate_baseline_pipeline.py
!python scripts/validate_research_execution.py

In [ ]:
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/FW-LNSA-NIDS')
DATA_ROOT = DRIVE_ROOT / 'data'
RESULT_ROOT = DRIVE_ROOT / 'results'
RESULT_ROOT.mkdir(parents=True, exist_ok=True)
print('Data root:', DATA_ROOT)
print('Persistent result root:', RESULT_ROOT)

## Required files

Place `KDDTrain+.txt` and `KDDTest+.txt` in `MyDrive/FW-LNSA-NIDS/data/nsl_kdd/`.

In [ ]:
NSL_TRAIN = DATA_ROOT / 'nsl_kdd' / 'KDDTrain+.txt'
NSL_TEST = DATA_ROOT / 'nsl_kdd' / 'KDDTest+.txt'
assert NSL_TRAIN.exists(), NSL_TRAIN
assert NSL_TEST.exists(), NSL_TEST
print(NSL_TRAIN.stat().st_size, NSL_TEST.stat().st_size)

## Research run

The command resumes from saved run rows after a Colab disconnect. Do not use `--max-runs` for final research results.

In [ ]:
!python scripts/run_research_suite.py \
  --profile research \
  --stages nsl_kdd_fw_lnsa \
  --output-dir "{RESULT_ROOT}" \
  --nsl-train-file "{NSL_TRAIN}" \
  --nsl-test-file "{NSL_TEST}"

In [ ]:
import pandas as pd
result_file = RESULT_ROOT / 'tables' / 'nsl_kdd_fw_lnsa_results.csv'
results = pd.read_csv(result_file)
print(results.shape)
display(results.groupby(['method_name', 'feature_set'])[['f1', 'recall', 'fpr', 'total_time_sec']].agg(['mean', 'std']))